In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from torchvision import transforms
from src.config import DATA_ROOT, IMAGE_SIZE
from src.data.dataset import MVTecDataset


In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),
    transforms.ToTensor(),
])
mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])
category = "bottle"
train_dataset = MVTecDataset(DATA_ROOT, category, "train", transform=image_transform)

pixel_sum = torch.zeros(3)
pixel_sum_sq = torch.zeros(3)
num_pixels = 0
avg_image = torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE))
for image, *_ in train_dataset:
    pixel_sum += image.sum(dim=(1, 2))
    pixel_sum_sq += image.square().sum(dim=(1, 2))
    num_pixels += image.shape[1] * image.shape[2]
    avg_image += image
mean = pixel_sum / num_pixels
std = torch.sqrt((pixel_sum_sq / num_pixels) - mean.square())
avg_image /= len(train_dataset)
print(f"{category} train/good mean: {mean.tolist()}")
print(f"{category} train/good std: {std.tolist()}")
plt.figure(figsize=(6, 6))
plt.imshow(avg_image.permute(1, 2, 0).numpy())
plt.title("Mean train/good image")
plt.axis("off")
plt.show()


### EDA notes: normal-image appearance

The channel statistics are descriptive for this category and preprocessing. They may be useful for normalization, but they do not establish that ImageNet normalization is unsuitable. The mean image shows average alignment; quantify per-pixel variance before making claims about spatial stability or augmentation needs.

Whether bottleneck capacity affects defect reconstruction is a hypothesis for controlled experiments. The loss and bottleneck ablations should test it; the mask-area distribution alone cannot establish that a narrower bottleneck improves detection.


In [ ]:
mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])
test_dataset = MVTecDataset(DATA_ROOT, category, "test", transform=image_transform, mask_transform=mask_transform)
cumulative_mask = torch.zeros((IMAGE_SIZE, IMAGE_SIZE))
mask_areas_pct = []
defect_counts = {}
for _, mask, label, path in test_dataset:
    defect_type = Path(path).parent.name
    defect_counts[defect_type] = defect_counts.get(defect_type, 0) + 1
    if int(label.item()) == 1:
        binary_mask = mask.squeeze(0) > 0
        cumulative_mask += binary_mask
        mask_areas_pct.append(float(binary_mask.float().mean() * 100))
areas = np.asarray(mask_areas_pct)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
im = axes[0].imshow(cumulative_mask.numpy(), cmap="inferno")
axes[0].set_title("Cumulative test-mask locations")
axes[0].axis("off")
fig.colorbar(im, ax=axes[0], shrink=0.7)
axes[1].hist(areas, bins=15, color="teal", edgecolor="black")
axes[1].set_title("Defect area per anomalous image")
axes[1].set_xlabel("Image area (%)")
axes[1].set_ylabel("Images")
axes[2].bar(defect_counts.keys(), defect_counts.values(), color="coral", edgecolor="black")
axes[2].set_title("Test images by defect type")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
print(f"Defect area (%): mean={areas.mean():.2f}, median={np.median(areas):.2f}, "
      f"q25={np.percentile(areas, 25):.2f}, q75={np.percentile(areas, 75):.2f}, "
      f"min={areas.min():.2f}, max={areas.max():.2f}")


### EDA notes: test defects

The earlier bottle summary reported a mean mask area of about 8.17% (minimum 0.82%, maximum 29%). An 8% mean is a substantial image region, not a “small defect” result. Rerun the cell above to confirm the current resized-mask statistics and inspect the median and quartiles: the mean alone does not show that most defects are small.

The cumulative mask is an image-coordinate frequency map. It can reveal recurring locations, but apparent edge concentration should be checked against object alignment and defect type before concluding that the model has a rim-specific failure. The evaluation report now measures border false positives and recall by defect area. Those are descriptive test results; do not tune the method from them.

The proposed link between defect size and bottleneck width remains a hypothesis until supported by the controlled ablations in `docs/experiments.md`.
